In [2]:
import pandas as pd
import json
import re

In [18]:
# Read in calculated top1 file for bts model
top1 = pd.read_csv("../aircraft_er_predictions/make_model_bts_top1_right.csv", sep=",")
top1

,left,right,match,match_confidence,left_id,right_id,p_match,L_make,L_model,L_name,R_description
0,COL make VAL BOEING COL model VAL 737 COL name...,COL description VAL Boeing 737-700/700LR/Max 7,1,0.990950,BOEING 737,612,0.990950,BOEING,737,"SURVEILLER, POSEIDON, MAX 8, MAX 9, MAX 7, MAX...",Boeing 737-700/700LR/Max 7
1,COL make VAL BOEING COL model VAL 787 COL name...,COL description VAL Boeing 787-10 Dreamliner,1,0.990946,BOEING 787,837,0.990946,BOEING,787,DREAMLINER,Boeing 787-10 Dreamliner
2,COL make VAL BOEING COL model VAL 747 COL name...,COL description VAL Boeing 747-100,1,0.989937,BOEING 747,816,0.989937,BOEING,747,"INTERCONTINENTAL, DREAMLIFTER, SHUTTLE CARRIER",Boeing 747-100
3,COL make VAL BOEING COL model VAL 747 COL name...,COL description VAL Boeing 747-400,1,0.989930,BOEING 747,819,0.989930,BOEING,747,"INTERCONTINENTAL, DREAMLIFTER, SHUTTLE CARRIER",Boeing 747-400
4,COL make VAL BOEING COL model VAL 737 COL name...,COL description VAL Boeing 737-400,1,0.989911,BOEING 737,617,0.989911,BOEING,737,"SURVEILLER, POSEIDON, MAX 8, MAX 9, MAX 7, MAX...",Boeing 737-400
...,...,...,...,...,...,...,...,...,...,...,...
229,COL make VAL AIRBUS COL model VAL A318 COL nam...,COL description VAL Airbus Industrie A-318,1,0.529505,AIRBUS A318,644,0.529505,AIRBUS,A318,ELITE,Airbus Industrie A-318
230,COL make VAL ANTONOV COL model VAL AN12,COL description VAL Antonov 24/26/32,1,0.435942,ANTONOV AN12,444,0.435942,ANTONOV,AN12,NaN,Antonov 24/26/32
231,COL make VAL BOMBARDIER COL model VAL BD500 CO...,COL description VAL A200-100 BD-500-1A10,1,0.416387,BOMBARDIER BD500,723,0.416387,BOMBARDIER,BD500,"CS100, CS300, A220-100, A220-300",A200-100 BD-500-1A10
232,COL make VAL GULFSTREAM COL model VAL GIV COL ...,COL description VAL Gulfstream III/V/ G-V Exec...,1,0.375498,GULFSTREAM GIV,667,0.375498,GULFSTREAM,GIV,"G300, G400, G350, G450",Gulfstream III/V/ G-V Exec/ G-5/550


In [27]:
#Read ground truth file
gt = pd.read_csv("../../aircraft_er/data/input/bts_labels_v1a.csv", sep=",")

#GT has a missing id (not mapped)
gt = gt[gt["right_id"].notna()]
#GT has duplicate rows
gt = gt.drop_duplicates()
gt


,left_id,right_id,description
0,IAI 1124,643.0,1124A Westwind II
1,BOMBARDIER BD500,723.0,A200-100 BD-500-1A10
2,BOMBARDIER BD500,724.0,A220-300 BD-500-1A11
3,SUD AVIATION SE210,680.0,Aerospatiale Caravelle SE-210
4,ATR ATR42,441.0,Aerospatiale/Aeritalia ATR-42
...,...,...,...
247,SHORT SC7,486.0,Shorts Harland SC-7 Skyvan
248,SIKORSKY S76,390.0,Sikorsky S-76
249,SOCATA TBM700,431.0,Socata TBM850
250,SWEARINGEN SA226,467.0,Swearingen Metro III


In [54]:
merged_model = gt.merge(top1, on= "right_id", how = "left", suffixes=("_gt", "_top1"))
merged_model["TP"] = (merged_model["left_id_gt"]==merged_model["left_id_top1"]).astype(int)
merged_model["FN"] = (merged_model["left_id_top1"].isna()).astype(int)
merged_model["FP"] = (merged_model["left_id_top1"].notna() & (merged_model["left_id_gt"]!=merged_model["left_id_top1"]) ).astype(int)
merged_model

,left_id_gt,right_id,description,left,right,match,match_confidence,left_id_top1,p_match,L_make,L_model,L_name,R_description,TP,FN,FP
0,IAI 1124,643.0,1124A Westwind II,COL make VAL IAI COL model VAL 1124 COL name V...,COL description VAL 1124A Westwind II,1.0,0.984116,IAI 1124,0.984116,IAI,1124,"WESTWIND II, WESTWIND I, SEA SCAN",1124A Westwind II,1,0,0
1,BOMBARDIER BD500,723.0,A200-100 BD-500-1A10,COL make VAL BOMBARDIER COL model VAL BD500 CO...,COL description VAL A200-100 BD-500-1A10,1.0,0.416387,BOMBARDIER BD500,0.416387,BOMBARDIER,BD500,"CS100, CS300, A220-100, A220-300",A200-100 BD-500-1A10,1,0,0
2,BOMBARDIER BD500,724.0,A220-300 BD-500-1A11,COL make VAL BOMBARDIER COL model VAL BD500 CO...,COL description VAL A220-300 BD-500-1A11,1.0,0.650973,BOMBARDIER BD500,0.650973,BOMBARDIER,BD500,"CS100, CS300, A220-100, A220-300",A220-300 BD-500-1A11,1,0,0
3,SUD AVIATION SE210,680.0,Aerospatiale Caravelle SE-210,COL make VAL SUD AVIATION COL model VAL SE210 ...,COL description VAL Aerospatiale Caravelle SE-210,1.0,0.941255,SUD AVIATION SE210,0.941255,SUD AVIATION,SE210,CARAVELLE,Aerospatiale Caravelle SE-210,1,0,0
4,ATR ATR42,441.0,Aerospatiale/Aeritalia ATR-42,COL make VAL ATR COL model VAL ATR42 COL name ...,COL description VAL Aerospatiale/Aeritalia ATR-42,1.0,0.970744,ATR ATR42,0.970744,ATR,ATR42,600 VERSION,Aerospatiale/Aeritalia ATR-42,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
235,SHORT SC7,486.0,Shorts Harland SC-7 Skyvan,COL make VAL SHORT COL model VAL SC7 COL name ...,COL description VAL Shorts Harland SC-7 Skyvan,1.0,0.980732,SHORT SC7,0.980732,SHORT,SC7,SKYVAN,Shorts Harland SC-7 Skyvan,1,0,0
236,SIKORSKY S76,390.0,Sikorsky S-76,COL make VAL SIKORSKY COL model VAL S76 COL na...,COL description VAL Sikorsky S-76,1.0,0.935704,SIKORSKY S76,0.935704,SIKORSKY,S76,SPIRIT,Sikorsky S-76,1,0,0
237,SOCATA TBM700,431.0,Socata TBM850,COL make VAL SOCATA COL model VAL TBM700 COL n...,COL description VAL Socata TBM850,1.0,0.973192,SOCATA TBM700,0.973192,SOCATA,TBM700,"TBM850, TBM900, TBM910, TBM930, TBM850 G1000",Socata TBM850,1,0,0
238,SWEARINGEN SA226,467.0,Swearingen Metro III,COL make VAL SWEARINGEN COL model VAL SA226 CO...,COL description VAL Swearingen Metro III,1.0,0.978426,SWEARINGEN SA226,0.978426,SWEARINGEN,SA226,"MERLIN III, MERLIN IIIB, METRO II",Swearingen Metro III,1,0,0


In [57]:
summary_model = merged_model[["TP", "FP", "FN"]].sum()
summary_model["unique_right_id"] = merged_model["right_id"].nunique(dropna=True)
summary_model["precision"] = summary_model["TP"]/(summary_model["TP"]+summary_model["FP"])
summary_model["recall"] = summary_model["TP"]/(summary_model["TP"]+summary_model["FN"])
summary_model["f1"] = 2*(summary_model["precision"]*summary_model["recall"] )/(summary_model["precision"]+summary_model["recall"] )
summary_model

TP                 228.000000
FP                   6.000000
FN                   6.000000
unique_right_id    240.000000
precision            0.974359
recall               0.974359
f1                   0.974359
dtype: float64

In [59]:
errors_merge_model = merged_model[merged_model["left_id_gt"]!=merged_model["left_id_top1"]]
print("model errors: ", len(errors_merge_model))

errors_merge_model

model errors:  12


,left_id_gt,right_id,description,left,right,match,match_confidence,left_id_top1,p_match,L_make,L_model,L_name,R_description,TP,FN,FP
9,AIRBUS A350,836.0,Airbus 350-1000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,1,0
34,ANTONOV AN124,880.0,Antonov 124,COL make VAL ANTONOV COL model VAL AN225 COL n...,COL description VAL Antonov 124,1.0,0.883874,ANTONOV AN225,0.883874,ANTONOV,AN225,MRIYA,Antonov 124,0,0,1
36,ANTONOV AN24,444.0,Antonov 24/26/32,COL make VAL ANTONOV COL model VAL AN12,COL description VAL Antonov 24/26/32,1.0,0.435942,ANTONOV AN12,0.435942,ANTONOV,AN12,NaN,Antonov 24/26/32,0,0,1
50,BEECH 200,458.0,Beechcraft Super King Air,COL make VAL BEECH COL model VAL 100 COL name ...,COL description VAL Beechcraft Super King Air,1.0,0.963645,BEECH 100,0.963645,BEECH,100,"KING AIR, SUPER KING AIR",Beechcraft Super King Air,0,0,1
149,EMBRAER ERJ190,678.0,Embraer 190,COL make VAL EMBRAER COL model VAL EMB135 COL ...,COL description VAL Embraer 190,1.0,0.648831,EMBRAER EMB135,0.648831,EMBRAER,EMB135,"LEGACY 600, ERJ135ER, ERJ140ER, ERJ140LR, ERJ1...",Embraer 190,0,0,1
159,EMBRAER EMB145,675.0,Embraer-145,COL make VAL EMBRAER COL model VAL EMB135 COL ...,COL description VAL Embraer-145,1.0,0.875952,EMBRAER EMB135,0.875952,EMBRAER,EMB135,"LEGACY 600, ERJ135ER, ERJ140ER, ERJ140LR, ERJ1...",Embraer-145,0,0,1
160,EMBRAER ERJ170,677.0,Embraer-Emb-170,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,1,0
180,GULFSTREAM GV,667.0,Gulfstream III/V/ G-V Exec/ G-5/550,COL make VAL GULFSTREAM COL model VAL GIV COL ...,COL description VAL Gulfstream III/V/ G-V Exec...,1.0,0.375498,GULFSTREAM GIV,0.375498,GULFSTREAM,GIV,"G300, G400, G350, G450",Gulfstream III/V/ G-V Exec/ G-5/550,0,0,1
186,ILYUSHIN IL96,881.0,Ilyushiin Il-96-400t,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,1,0
187,ILYUSHIN IL76,877.0,Ilyushin 76/TD,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,1,0


In [60]:

bts_block = pd.read_csv("../aircraft_er_predictions/eval_make_model_bts_block_make_model_make_model_union_top1_right.csv", sep=",")
merged_bts_block = gt.merge(bts_block, on= "right_id", how = "left", suffixes=("_gt", "_top1"))
merged_bts_block["TP"] = (merged_bts_block["left_id_gt"]==merged_bts_block["left_id_top1"]).astype(int)
merged_bts_block["FN"] = (merged_bts_block["left_id_top1"].isna()).astype(int)
merged_bts_block["FP"] = (merged_bts_block["left_id_top1"].notna() & (merged_bts_block["left_id_gt"]!=merged_bts_block["left_id_top1"]) ).astype(int)
merged_bts_block.head()


,left_id_gt,right_id,description,left,right,match,match_confidence,left_id_top1,p_match,L_make,L_model,L_name,R_description,TP,FN,FP
0,IAI 1124,643.0,1124A Westwind II,COL make VAL IAI COL model VAL 1124 COL name V...,COL description VAL 1124A Westwind II\n,1.0,0.999997,IAI 1124,0.999997,IAI,1124,WESTWIND 2 WESTWIND 1,1124A Westwind II,1,0,0
1,BOMBARDIER BD500,723.0,A200-100 BD-500-1A10,COL make VAL BOMBARDIER COL model VAL BD500,COL description VAL A200-100 BD-500-1A10\n,1.0,0.999997,BOMBARDIER BD500,0.999997,BOMBARDIER,BD500,NaN,A200-100 BD-500-1A10,1,0,0
2,BOMBARDIER BD500,724.0,A220-300 BD-500-1A11,COL make VAL BOMBARDIER COL model VAL BD500,COL description VAL A220-300 BD-500-1A11\n,1.0,0.999997,BOMBARDIER BD500,0.999997,BOMBARDIER,BD500,NaN,A220-300 BD-500-1A11,1,0,0
3,SUD AVIATION SE210,680.0,Aerospatiale Caravelle SE-210,COL make VAL SUD AVIATION COL model VAL SE210 ...,COL description VAL Aerospatiale Caravelle SE-...,1.0,0.999997,SUD AVIATION SE210,0.999997,SUD AVIATION,SE210,CARAVELLE,Aerospatiale Caravelle SE-210,1,0,0
4,ATR ATR42,441.0,Aerospatiale/Aeritalia ATR-42,COL make VAL ATR COL model VAL ATR42,COL description VAL Aerospatiale/Aeritalia ATR...,1.0,0.999997,ATR ATR42,0.999997,ATR,ATR42,NaN,Aerospatiale/Aeritalia ATR-42,1,0,0


In [61]:
summary_bts_block = merged_bts_block[["TP", "FP", "FN"]].sum()
summary_bts_block["unique_right_id"] = merged_bts_block["right_id"].nunique(dropna=True)
summary_bts_block["precision"] = summary_bts_block["TP"]/(summary_bts_block["TP"]+summary_bts_block["FP"])
summary_bts_block["recall"] = summary_bts_block["TP"]/(summary_bts_block["TP"]+summary_bts_block["FN"])
summary_bts_block["f1"] = 2*(summary_bts_block["precision"]*summary_bts_block["recall"] )/(summary_bts_block["precision"]+summary_bts_block["recall"] )
summary_bts_block

TP                 166.000000
FP                  40.000000
FN                  34.000000
unique_right_id    240.000000
precision            0.805825
recall               0.830000
f1                   0.817734
dtype: float64

In [80]:

merged2.to_csv('temp.csv', index=False)

In [62]:
bts = pd.read_csv("../aircraft_er_predictions/eval_make_model_bts_model_make_model_union_top1_right.csv", sep=",")
merged_bts = gt.merge(bts, on= "right_id", how = "left", suffixes=("_gt", "_top1"))
merged_bts["TP"] = (merged_bts["left_id_gt"]==merged_bts["left_id_top1"]).astype(int)
merged_bts["FN"] = (merged_bts["left_id_top1"].isna()).astype(int)
merged_bts["FP"] = (merged_bts["left_id_top1"].notna() & (merged_bts["left_id_gt"]!=merged_bts["left_id_top1"]) ).astype(int)
merged_bts.head()



,left_id_gt,right_id,description,left,right,match,match_confidence,left_id_top1,p_match,L_make,L_model,L_name,R_description,TP,FN,FP
0,IAI 1124,643.0,1124A Westwind II,COL make VAL IAI COL model VAL 1124 COL name V...,COL description VAL 1124A Westwind II\n,1.0,0.999997,IAI 1124,0.999997,IAI,1124,WESTWIND 2 WESTWIND 1,1124A Westwind II,1,0,0
1,BOMBARDIER BD500,723.0,A200-100 BD-500-1A10,COL make VAL BOMBARDIER COL model VAL BD500,COL description VAL A200-100 BD-500-1A10\n,1.0,0.999997,BOMBARDIER BD500,0.999997,BOMBARDIER,BD500,NaN,A200-100 BD-500-1A10,1,0,0
2,BOMBARDIER BD500,724.0,A220-300 BD-500-1A11,COL make VAL BOMBARDIER COL model VAL BD500,COL description VAL A220-300 BD-500-1A11\n,1.0,0.999996,BOMBARDIER BD500,0.999996,BOMBARDIER,BD500,NaN,A220-300 BD-500-1A11,1,0,0
3,SUD AVIATION SE210,680.0,Aerospatiale Caravelle SE-210,COL make VAL SUD AVIATION COL model VAL SE210 ...,COL description VAL Aerospatiale Caravelle SE-...,1.0,0.999997,SUD AVIATION SE210,0.999997,SUD AVIATION,SE210,CARAVELLE,Aerospatiale Caravelle SE-210,1,0,0
4,ATR ATR42,441.0,Aerospatiale/Aeritalia ATR-42,COL make VAL ATR COL model VAL ATR42,COL description VAL Aerospatiale/Aeritalia ATR...,1.0,0.999997,ATR ATR42,0.999997,ATR,ATR42,NaN,Aerospatiale/Aeritalia ATR-42,1,0,0


In [63]:
summary_bts = merged_bts[["TP", "FP", "FN"]].sum()
summary_bts["unique_right_id"] = merged_bts["right_id"].nunique(dropna=True)
summary_bts["precision"] = summary_bts["TP"]/(summary_bts["TP"]+summary_bts["FP"])
summary_bts["recall"] = summary_bts["TP"]/(summary_bts["TP"]+summary_bts["FN"])
summary_bts["f1"] = 2*(summary_bts["precision"]*summary_bts["recall"] )/(summary_bts["precision"]+summary_bts["recall"] )
summary_bts

TP                 154.000000
FP                  82.000000
FN                   4.000000
unique_right_id    240.000000
precision            0.652542
recall               0.974684
f1                   0.781726
dtype: float64